In [9]:
import os
import pandas as pd

config

In [10]:
DATA_DIR = "../data"
INPUT_FILE = os.path.join(DATA_DIR, "twcs.csv")
OUTPUT_FILE = os.path.join(DATA_DIR, "data.csv")
BRAND_HANDLE = "AmericanAir"  

Load

In [11]:
dtype_map = {
    "tweet_id": str,
    "author_id": str,
    "in_response_to_tweet_id": str,
    "response_tweet_id": str,
}
df = pd.read_csv(INPUT_FILE, dtype=dtype_map)
df.columns = df.columns.str.strip()  # guard against stray whitespace in headers

print(f"Loaded {len(df):,} total rows")

Loaded 2,811,774 total rows


Build reply-chain lookups (one pass over the full data)

In [12]:
parent_of = df.set_index("tweet_id")["in_response_to_tweet_id"].to_dict()

children_of = {}
for tid, resp in zip(df["tweet_id"], df["response_tweet_id"]):
    if pd.notna(resp):
        for child_id in str(resp).split(","):
            children_of.setdefault(tid, []).append(child_id.strip())

BFS outward from every AmericanAir tweet to pull whole threads

In [15]:
seeds = set(df.loc[df["author_id"] == BRAND_HANDLE, "tweet_id"])
print(f"AmericanAir tweets found: {len(seeds):,}")

visited = set(seeds)
frontier = set(seeds)

while frontier:
    next_frontier = set()
    for tid in frontier:
        parent = parent_of.get(tid)
        if pd.notna(parent) and parent not in visited:
            next_frontier.add(parent)
        for child_id in children_of.get(tid, []):
            if child_id not in visited:
                next_frontier.add(child_id)
    visited |= next_frontier
    frontier = next_frontier

print(f"Full thread rows (incl. customer side): {len(visited):,}")

AmericanAir tweets found: 36,764
Full thread rows (incl. customer side): 97,206


In [16]:
result = df[df["tweet_id"].isin(visited)].copy()
result["created_at"] = pd.to_datetime(result["created_at"], errors="coerce")
result = result.sort_values("created_at").reset_index(drop=True)

os.makedirs(DATA_DIR, exist_ok=True)
result.to_csv(OUTPUT_FILE, index=False)

C:\Users\HARIHARAN\AppData\Local\Temp\ipykernel_25920\1709751213.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  result["created_at"] = pd.to_datetime(result["created_at"], errors="coerce")


In [17]:
print(f"Saved {len(result):,} rows to {OUTPUT_FILE}")
print(result["inbound"].value_counts(), sep="\n")  # should see both True and False
print(result.head())

Saved 87,584 rows to ../data\data.csv
inbound
True     50054
False    37530
Name: count, dtype: int64
  tweet_id    author_id  inbound                created_at  \
0   626918       268460     True 2011-06-16 13:38:36+00:00   
1   626916  AmericanAir    False 2011-06-16 13:49:04+00:00   
2  2502768       713819     True 2013-09-04 18:22:07+00:00   
3  2502766  AmericanAir    False 2013-09-04 18:33:04+00:00   
4  1023937  AmericanAir    False 2014-11-14 04:21:36+00:00   

                                                text response_tweet_id  \
0  @americanair Do you have a number to call you ...            626916   
1  @28752 Text "FLYAA" (35922) with your flight n...            626917   
2  Personally @97121 I think @AmericanAir needs a...           2502766   
3  @713819 That sounds like fun, Sunita! We can a...           2502767   
4  @362312 We don't like nightmare's, Carson. Has...           1023936   

  in_response_to_tweet_id  
0                     NaN  
1                  62691